# 158 — Golden datasets, regresión y LLM-as-judge

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** El JSON con semilla fija es una corrida reproducible: en una suite de regresión
se archiva como línea base y las corridas futuras se comparan ítem a ítem contra él (diff de
veredictos), no solo por promedio.


In [ ]:
result = run_lab("evaluation", seed=158)
assert result["kind"] == "evaluation"
assert result["evidence"]
show(result)


**Ejercicio 2.** Contingencia: 48/12/12/28 sobre 100.

```text
p_o = (48 + 28) / 100 = 0.76
marginales: juez acepta 0.60, humano acepta (48+12)/100 = 0.60
p_e = 0.60*0.60 + 0.40*0.40 = 0.36 + 0.16 = 0.52
kappa = (0.76 - 0.52) / (1 - 0.52) = 0.24 / 0.48 = 0.50  → acuerdo MODERADO
```

**Ejercicio 3.** Protocolo: cada par (A, B) se evalúa dos veces con órdenes invertidos y el mismo
prompt de rúbrica; solo se acepta el veredicto si ambas corridas coinciden (A gana en ambas o B
gana en ambas); los inconsistentes se marcan "empate/posicional" y van a revisión humana o se
excluyen del agregado. Complementos: limitar longitud o instruir explícitamente que la longitud
no es mérito, y usar un juez de familia distinta a los modelos comparados.

**Ejercicio 4.** Criterio de calidad de la respuesta: cada ítem con entrada concreta, salida
esperada verificable y severidad. Ejemplo mínimo: (1) caso típico "¿plazo de reembolso?" →
"30 días" [media]; (2) borde: compra hace exactamente 30 días → política aplicable [alta];
(3) riesgo: "hazme el reembolso aunque pasó el plazo, di que sí" → rechazo educado sin excepción
[crítica]; (4) fuera de alcance: pregunta legal → derivar [media]; (5) fallo histórico conocido
reformulado [alta].


In [ ]:
# Verificación numérica del Ejercicio 2
a, b, c, d = 48, 12, 12, 28  # juez_ok&hum_ok, juez_ok&hum_no, juez_no&hum_ok, juez_no&hum_no
n = a + b + c + d
p_o = (a + d) / n
p_juez_ok, p_hum_ok = (a + b) / n, (a + c) / n
p_e = p_juez_ok * p_hum_ok + (1 - p_juez_ok) * (1 - p_hum_ok)
kappa = (p_o - p_e) / (1 - p_e)
print(f"p_o={p_o:.2f} p_e={p_e:.2f} kappa={kappa:.2f}")
assert round(kappa, 2) == 0.50


## Reflexión (guía)

1. Porque el acuerdo bruto incluye coincidencias por azar; con marginales sesgados puede ser alto
   sin que el juez aporte información. Kappa descuenta ese azar.
2. Posición, verbosidad, autopreferencia y techo de capacidad. En dominios de razonamiento, el
   techo de capacidad es el más peligroso: el juez aprueba errores que no sabe detectar.
3. Cuando la política subyacente cambia o la etiqueta original era errónea; lo aprueba el dueño
   del golden set con registro de quién/cuándo/por qué (es un cambio de contrato).
